In [ ]:
# import necessary libraries
import pandas as pd
import numpy as np
import tqdm

# import custom modules
import sys
sys.path.append("..")
from custom_modules.feature_engineering import (
    FeatureGeneratorLevel1,
    FeatureGeneratorLevel2,
    FeatureGeneratorLevel3
)
from custom_modules.preprocessing import Preprocessor

In [ ]:
# load the training data from data/train.csv
df = pd.read_csv("../data/train.csv")
df.drop(columns=["id"], inplace=True)

In [ ]:
# separate the features and target variable
X = df.drop(columns=["Heart Disease"])
y = df["Heart Disease"]

In [ ]:
# label encode the target variable
y = y.map({"Absence": 0, "Presence": 1})

In [ ]:
# generate level 1 features for the training data
fgl1 = FeatureGeneratorLevel1()
X_level_1 = fgl1.generate(X)

X_level_1.head()

In [ ]:
# merge the level 1 features with baseline features
X = pd.concat([X, X_level_1], axis=1)

In [ ]:
# split the data into training and validation sets using stratified sampling
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# create a pipeline with the preprocessor and logistic regression model with scaling as True
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
model1 = Pipeline([
    ("preprocessor", Preprocessor(scaling=True)),
    ("model", LogisticRegression(random_state=42))
])

In [ ]:
# train model1 on the training data
model1.fit(X_train, y_train)

In [ ]:
# predict on the validation set using model1
y_pred = model1.predict(X_val)

In [ ]:
# check model1 performance on the validation set
from sklearn.metrics import classification_report
print(classification_report(y_val, y_pred))

In [ ]:
# check model1 roc auc score on the validation set
from sklearn.metrics import roc_auc_score
y_pred_proba = model1.predict_proba(X_val)[:, 1]
print("Model1 ROC AUC Score:", roc_auc_score(y_val, y_pred_proba))

In [ ]:
# create a hyperparameter grid for model1
param_grid = [
    # L2 Regularization (Most stable baseline)
    {
        "model__penalty": ["l2"],
        "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
        "model__solver": ["lbfgs", "saga"],
        "model__max_iter": [1000]
    },

    # L1 Regularization (Feature selection)
    {
        "model__penalty": ["l1"],
        "model__C": [0.001, 0.01, 0.1, 1, 10],
        "model__solver": ["saga"],
        "model__max_iter": [2000]
    },

    # ElasticNet (Hybrid regularization)
    {
        "model__penalty": ["elasticnet"],
        "model__C": [0.01, 0.1, 1, 10],
        "model__solver": ["saga"],
        "model__l1_ratio": [0.2, 0.5, 0.8],
        "model__max_iter": [2000]
    }
]


In [ ]:
# apply GridSearchCV to model1 using the hyperparameter grid and stratified 3-fold cross validation with roc_auc as the scoring metric
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.model_selection import GridSearchCV

stratified_kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=model1,
    param_grid=param_grid,
    cv=stratified_kfold,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=2
)

In [ ]:
# fit the grid search to the training data
grid_search.fit(X_train, y_train)

In [ ]:
# print the best hyperparameters and corresponding roc auc score from the grid search
print("Best Hyperparameters:", grid_search.best_params_)
print("Best ROC AUC Score:", grid_search.best_score_)

In [ ]:
# save the best model from the grid search as model2
model2 = grid_search.best_estimator_

In [ ]:
# predict on the validation set using model2
y_pred = model2.predict(X_val)

In [ ]:
# check model2 performance on the validation set
print(classification_report(y_val, y_pred))

In [ ]:
# check model2 roc auc score on the validation set
y_pred_proba = model2.predict_proba(X_val)[:, 1]

In [ ]:
# load the test data from data/test.csv
test_df = pd.read_csv("../data/test.csv")

In [ ]:
# seperate the id column and features from the test data
test_ids = test_df["id"]
X_test = test_df.drop(columns=["id"])

In [ ]:
# generate level 1 features for the test data and merge them with the baseline features
X_test_level_1 = fgl1.generate(X_test)
X_test = pd.concat([X_test, X_test_level_1], axis=1)
X_test.head()

In [ ]:
# predict probability of heart disease on the test data using model2
test_preds_proba = model2.predict_proba(X_test)[:, 1]

In [ ]:
# save the test predictions in a csv file with columns id and Heart Disease in the output folder
submission_df = pd.DataFrame({
    "id": test_ids,
    "Heart Disease": test_preds_proba
})
submission_df.to_csv("../output/submission.csv", index=False)